# 08 · 학습 전에 "작은 구조(cystic_duct)를 잡을 수 있는가" 검증

> **공부 기록 노트북 8번.** 첫 `sam2_temporal` 학습은 `cystic_duct` Dice 가
> **0.000** 이었습니다. 원인을 코드에서 고쳤지만(샘플러·early-stop·loss),
> **수 시간짜리 본학습을 돌리기 전에** "이 수정으로 duct 가 실제로 잡힐
> 가능성이 있는가"를 *몇 분 안에* 확인하는 것이 이 노트북의 목적입니다.

## 무엇을 검증하나 (본학습 없이)

| # | 검증 | 답하는 질문 |
|---|---|---|
| 1 | **loss 설계** (합성 예제) | Focal-Tversky 가 *놓침(FN)* 을 Dice 보다 더 세게 벌하는가? |
| 2 | **라벨 진단** | duct 가 애초에 라벨돼 있는가? 얼마나 희소한가? |
| 3 | **샘플러 효과** | 가중 샘플러가 duct 프레임 노출을 실제로 몇 배 올리는가? |
| 4 | **단일 배치 과적합** (핵심) | 모델+loss+라벨이 duct 를 *표현할 수 있는가*? |

**핵심 논리 (4번):** 한 배치에 대해 과적합시켰을 때 duct Dice 가 높이 올라가면,
모델·loss·라벨은 duct 를 **표현할 능력이 있다** → 본학습의 과제는 *일반화*이지
*능력*이 아닙니다. 반대로 한 배치조차 못 올리면, 스케줄을 늘릴 게 아니라
**라벨 품질이나 구조(architecture)** 를 의심해야 합니다.

⚠️ 점수의 절대값이 목적이 아닙니다 — *"0 을 벗어날 수 있는가"* 의 신호를 봅니다.

## 0. 환경 준비

Colab 은 매번 빈 컴퓨터입니다. 코드를 받고 라이브러리를 설치합니다.

> 이 노트북이 검증하는 수정(Focal-Tversky loss, clip-level 샘플러)은 아직
> `main` 에 머지되지 않았을 수 있으니, 아래 `BRANCH` 를 기능 브랜치로 둡니다.
> 머지된 뒤에는 `main` 으로 바꾸세요.

In [ ]:
%cd /content
import os
BRANCH = "claude/stoic-brown-jllvfb"   # 머지 후 "main" 으로
if not os.path.isdir("surgical-ai"):
    !git clone -b $BRANCH https://github.com/duck-bin/surgical-ai.git
%cd surgical-ai
!git fetch origin $BRANCH && git checkout $BRANCH && git pull --ff-only origin $BRANCH
!bash scripts/colab_setup.sh

import torch
print("준비 완료 ·", os.getcwd())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "없음 → Runtime > Change runtime type > GPU 권장 (4번 과적합이 느려짐)")

## 1. loss 설계 검증 — Focal-Tversky 는 *놓침* 을 더 세게 벌하는가?

데이터 없이 합성 예제로 먼저 확인합니다. 얇은 2픽셀짜리 가짜 'duct' 줄무늬를
정답으로 두고, 세 가지 예측을 비교합니다:

- **good** : 정확히 맞춤
- **miss (FN)** : duct 를 통째로 놓침 (배경으로 예측) — *작은 구조의 전형적 실패*
- **over (FP)** : duct 는 맞추되, 같은 픽셀 수만큼 엉뚱한 곳에 더 그림

Dice 는 FN·FP 를 **대칭**으로 보지만, Tversky(β>α) 는 **FN(놓침)** 을 더
무겁게 벌해야 합니다 — 그게 recall 압력의 핵심입니다.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.losses.dice import DiceLoss
from src.losses.tversky import TverskyLoss

NUM_CLASSES, SIZE, DUCT = 6, 32, 3

def logits_from(pred):
    oh = torch.nn.functional.one_hot(pred, NUM_CLASSES).permute(0, 3, 1, 2).float()
    return oh * 20.0 - 10.0   # ~hard softmax

target = torch.zeros(1, SIZE, SIZE, dtype=torch.long)
target[:, 14:16, :] = DUCT                       # thin 2-row duct stripe

good = target.clone()
miss = torch.zeros(1, SIZE, SIZE, dtype=torch.long)   # FN: duct 통째로 놓침
over = target.clone(); over[:, 0:2, :] = DUCT          # FP: 같은 면적 헛예측

losses = {
    "Dice":           DiceLoss(),
    "Tversky .3/.7":  TverskyLoss(alpha=0.3, beta=0.7, gamma=1.0),
    "FocalTv .3/.7":  TverskyLoss(alpha=0.3, beta=0.7, gamma=1.333),
}
cases = {"good": good, "miss (FN)": miss, "over (FP)": over}

print(f"{'loss':14s} " + "".join(f"{c:>12s}" for c in cases))
rows = {}
for lname, lf in losses.items():
    vals = [float(lf(logits_from(p), target)) for p in cases.values()]
    rows[lname] = vals
    print(f"{lname:14s} " + "".join(f"{v:12.3f}" for v in vals))

# miss/over 비율: 클수록 '놓침'을 상대적으로 더 벌함
print("\nmiss/over 벌점 비율 (클수록 recall 압력):")
for lname, vals in rows.items():
    print(f"  {lname:14s}: {vals[1]/max(vals[2],1e-6):.2f}x")

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(cases)); w = 0.26
for i, (lname, vals) in enumerate(rows.items()):
    ax.bar(x + (i-1)*w, vals, w, label=lname)
ax.set_xticks(x); ax.set_xticklabels(list(cases)); ax.set_ylabel("loss")
ax.set_title("작은 구조: Dice vs (Focal-)Tversky — 'miss(FN)' 막대를 보라")
ax.legend(); plt.tight_layout(); plt.show()

**읽는 법.** `miss (FN)` 막대가 `over (FP)` 보다 높을수록, 그리고 그 비율이
Dice 보다 Tversky/FocalTversky 에서 클수록 — loss 가 "작은 구조를 놓치는 것"을
더 강하게 벌하도록 설계된 것입니다. 이게 본학습에서 duct recall 을 끌어올리는
힘입니다. (수치는 `tests/test_losses.py` 가 회귀로 고정합니다.)

## 2. 라벨 진단 — duct 는 라벨돼 있는가? 얼마나 희소한가?

아무리 좋은 loss·샘플러도 **라벨이 없거나 엉망이면** 못 배웁니다. 본학습 전에
train 분할의 일부를 훑어 (1) 클래스별 픽셀 빈도, (2) duct 가 들어있는 프레임
비율, (3) duct 면적 분포를 봅니다.

> ⏱️ 전체 train 을 다 훑으면 느리므로 앞 `N_DIAG` 프레임만 봅니다 (예시 진단).
> 데이터가 아직 없으면 먼저 `!bash scripts/download_cholecseg8k.sh` 를 실행하세요.

In [ ]:
from torch.utils.data import Subset
from src.data.cholecseg8k import CholecSeg8kDataset, CLASS_NAMES, NUM_CLASSES
from src.data.transforms import build_eval_transforms

IMG = 256
N_DIAG = 600     # 진단용 표본 (늘릴수록 정확·느림)

diag_ds = CholecSeg8kDataset(split="train", image_size=IMG,
                             transform=build_eval_transforms(IMG))
N = min(N_DIAG, len(diag_ds))
print(f"train 프레임 총 {len(diag_ds)} 중 앞 {N} 개로 진단\n")

counts = np.zeros(NUM_CLASSES, dtype=np.int64)
duct_frames, duct_area = 0, []
for i in range(N):
    m = diag_ds[i]["mask"].numpy()
    binc = np.bincount(m.ravel(), minlength=NUM_CLASSES)[:NUM_CLASSES]
    counts += binc
    a = int(binc[CLASS_NAMES.index("cystic_duct")])
    if a > 0:
        duct_frames += 1
        duct_area.append(a / m.size * 100)

freq = counts / counts.sum()
print(f"{'class':14s} {'pixel %':>9s}")
for name, f in zip(CLASS_NAMES, freq):
    print(f"{name:14s} {f*100:9.4f}")
print(f"\nduct 포함 프레임: {duct_frames}/{N} = {duct_frames/N*100:.1f}%")
if duct_area:
    da = np.array(duct_area)
    print(f"duct 면적(프레임 내 %): 중앙값 {np.median(da):.3f},  "
          f"최소 {da.min():.3f},  최대 {da.max():.3f}")
    plt.figure(figsize=(6, 3))
    plt.hist(da, bins=30, color="goldenrod")
    plt.xlabel("프레임 내 duct 면적 (%)"); plt.ylabel("프레임 수")
    plt.title("cystic_duct 는 얼마나 작은가"); plt.tight_layout(); plt.show()
else:
    print("⚠️ 표본에서 duct 픽셀을 못 찾음 — N_DIAG 를 늘리거나 라벨/색상맵 확인")

**읽는 법.** duct pixel % 가 0.0x 수준(수십분의 1 %)이고 "duct 포함 프레임"
비율도 낮다면 — 이게 본학습이 어려운 *근본 이유*입니다. 면적 히스토그램이
0 근처에 몰려 있으면 thin-structure 문제임이 확인됩니다. 만약 duct 픽셀이
**아예 0** 이면 색상맵(`CHOLECSEG8K_COLOR_MAP`) 이나 다운로드를 먼저 의심하세요.

## 3. duct 프레임 눈으로 보기 — 라벨이 입력과 맞는가?

duct 가 들어있는 프레임 몇 장을 `[입력 | duct 강조 오버레이]` 로 봅니다.
입력에 보이는 관(tube)이 노란 마스크와 대체로 일치하면 라벨이 신뢰할 만한
것이고, 입력엔 뚜렷한 duct 가 있는데 마스크가 비어 있으면 *라벨 누락*
(README 가 언급한 한계)을 눈으로 확인하는 것입니다.

In [ ]:
from src.utils.viz import overlay_mask

DUCT = CLASS_NAMES.index("cystic_duct")
idx = [i for i in range(N) if (diag_ds[i]["mask"] == DUCT).any()][:3]

if not idx:
    print("표본에 duct 프레임이 없음 — N_DIAG 를 늘리세요")
else:
    fig, ax = plt.subplots(len(idx), 2, figsize=(8, 4*len(idx)))
    ax = np.atleast_2d(ax)
    for r, i in enumerate(idx):
        s = diag_ds[i]
        rgb = s["image"].permute(1, 2, 0).numpy()
        rgb = ((rgb - rgb.min())/(rgb.max()-rgb.min()+1e-8)*255).astype("uint8")
        mask = s["mask"].numpy()
        duct_only = np.where(mask == DUCT, DUCT, 0)     # duct 만 강조
        ax[r, 0].imshow(rgb); ax[r, 0].set_title(f"input #{i}")
        ax[r, 1].imshow(overlay_mask(rgb, duct_only)); ax[r, 1].set_title("cystic_duct (GT)")
        for a in ax[r]: a.axis("off")
    plt.tight_layout(); plt.show()

## 4-a. 샘플러 효과 — duct 노출이 몇 배 늘어나는가?

첫 학습 실패의 큰 원인은 **temporal 모델에서 가중 샘플러가 꺼져 있어** duct 를
자연 빈도로만 봤다는 점이었습니다. 가중 샘플러가 duct 프레임을 뽑을 확률을
자연 빈도 대비 얼마나 끌어올리는지 표본에서 계산합니다.

In [ ]:
from src.data.class_balance import compute_class_stats

sub = Subset(diag_ds, list(range(N)))
_, sample_weights = compute_class_stats(sub, NUM_CLASSES)
duct_idx = [i for i in range(N) if (diag_ds[i]["mask"] == DUCT).any()]

natural = len(duct_idx) / N
weighted = sample_weights[duct_idx].sum() / sample_weights.sum()
print(f"duct 프레임을 뽑을 확률")
print(f"  자연 빈도        : {natural*100:6.2f}%")
print(f"  가중 샘플러      : {weighted*100:6.2f}%")
print(f"  → 노출 배수      : {weighted/max(natural,1e-9):5.2f}x")

## 4-b. clip-level 샘플러 — temporal 경로의 수정 확인

temporal 은 프레임이 아니라 **clip(윈도우)** 단위라, 프레임 가중치를 각 clip 의
*타깃(마지막) 프레임* 으로 옮기는 `window_sample_weights` 를 새로 넣었습니다.
개념을 작은 합성 예로 확인합니다 (본학습에선 실제 per-frame 가중치가 캐시에서
들어갑니다).

In [ ]:
from src.data.class_balance import window_sample_weights

# 프레임 3 이 rare 클래스를 포함한다고 하자 (가중치 9), 나머지는 1.
frame_w = np.array([1., 1., 1., 9., 1.])
windows = [(0,1,2), (1,2,3), (2,3,4)]   # 타깃 = 마지막 프레임
w = window_sample_weights(windows, frame_w)
for win, wi in zip(windows, w):
    print(f"clip {win} → 타깃 프레임 {win[-1]} → 가중치 {wi:.0f}")
print(f"\n→ rare 프레임(3)을 예측하는 clip 이 가장 자주 뽑힘: {w.argmax()==1}")

## 5. (핵심) 단일 배치 과적합 — 모델은 duct 를 표현할 수 있는가?

duct 가 들어있는 **한 배치**만 골라, 거기에 모델을 과적합시킵니다. 새 loss
(Focal + Focal-Tversky)로 최적화하면서 duct Dice 가 0 에서 올라가는지 봅니다.

- **올라가면** → 모델·loss·라벨은 duct 를 표현할 능력이 있음 → 본학습 GO.
- **안 올라가면** → 스케줄 문제가 아니라 라벨/구조 문제 → 그쪽을 먼저 고쳐야 함.

> 빠른 신호를 위해 가벼운 **U-Net** 으로 합니다 (GPU 에서 ~1–2분). 실제 연구
> 모델까지 확인하려면 `MODEL_CFG` 를 `configs/model/sam2_lora.yaml` 또는
> `configs/model/sam2_temporal.yaml` 로 바꾸세요 (느리지만 진짜 경로).

In [ ]:
from omegaconf import OmegaConf
from src.train.train_segmentation import build_model
from src.losses.focal import FocalLoss
from src.eval.metrics import dice_score

MODEL_CFG = "configs/model/unet_baseline.yaml"   # 빠른 신호. sam2_* 로 바꾸면 진짜 경로
device = "cuda" if torch.cuda.is_available() else "cpu"
K, STEPS = 6, 150

# duct 가 들어있는 프레임 K 장으로 한 배치 구성
batch_idx = [i for i in range(N) if (diag_ds[i]["mask"] == DUCT).any()][:K]
assert batch_idx, "duct 프레임을 못 찾음 — N_DIAG 를 늘리세요"
imgs = torch.stack([diag_ds[i]["image"] for i in batch_idx]).to(device)
masks = torch.stack([diag_ds[i]["mask"] for i in batch_idx]).to(device)
print(f"과적합 배치: {len(batch_idx)} 프레임, 모델 {OmegaConf.load(MODEL_CFG).name}, {device}")

model = build_model(OmegaConf.load(MODEL_CFG), pretrained=True).to(device).train()
focal = FocalLoss(gamma=2.0)
tversky = TverskyLoss(alpha=0.3, beta=0.7, gamma=1.333)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

hist = []
for step in range(STEPS):
    opt.zero_grad()
    logits = model(imgs)
    loss = focal(logits, masks) + tversky(logits, masks)
    loss.backward(); opt.step()
    if step % 10 == 0 or step == STEPS-1:
        with torch.no_grad():
            pred = logits.argmax(1)
            duct_dice = float(dice_score(pred, masks, NUM_CLASSES)[0][DUCT])
        hist.append((step, float(loss), duct_dice))
        print(f"step {step:3d}  loss {float(loss):.3f}  duct_dice {duct_dice:.3f}")

steps, lossv, ducts = zip(*hist)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(steps, lossv, "-o"); ax[0].set_title("loss"); ax[0].set_xlabel("step")
ax[1].plot(steps, ducts, "-o", color="goldenrod")
ax[1].set_title("cystic_duct Dice (한 배치 과적합)"); ax[1].set_xlabel("step")
ax[1].set_ylim(-0.02, 1.02); ax[1].axhline(0, color="gray", lw=0.5)
plt.tight_layout(); plt.show()

final = ducts[-1]
verdict = ("✅ duct 를 표현할 수 있음 → 본학습 GO" if final > 0.5 else
           "⚠️ 한 배치도 잘 안 오름 → 라벨/구조/하이퍼파라미터 점검 먼저")
print(f"\n최종 duct Dice (과적합): {final:.3f}\n{verdict}")

## 마무리 — 이 노트북이 말해주는 것

본학습(수 시간)을 돌리기 **전에** 빠르게 확인했습니다:

1. **loss** 가 작은 구조의 *놓침(FN)* 을 Dice 보다 세게 벌하는가 → 합성 예제로 확인
2. **라벨** 이 존재하고 얼마나 희소한가 → duct pixel % · 프레임 비율 · 면적 분포
3. **샘플러** 가 duct 노출을 몇 배 올리는가 → 프레임/clip 양쪽
4. **모델+loss+라벨** 이 duct 를 표현할 수 있는가 → 단일 배치 과적합 (핵심)

### 의사결정
- 5번 duct Dice 가 잘 오르면 → 본학습(`run_pipeline` / `train_segmentation
  model=sam2_temporal`)으로 진행하고, 거기선 *일반화* 를 본다 (val/test duct Dice).
- 안 오르면 → 더 긴 학습이 아니라 **라벨 sanity(2·3번)·구조·loss 비중** 을
  먼저 손본다. 노트북 01/07 의 라벨 검수와 연결.

> 이 노트북은 *가능성* 을 보는 사전 점검입니다. 최종 성능은 본학습의 val/test
> 곡선과 `notebooks/07_results_visualization.ipynb` 로 확인하세요.